In [1]:
pip install opencv-python-headless numpy easyocr pyspellchecker

In [7]:
pip install opencv-python-headless numpy torch transformers pillow

In [9]:
import cv2
import numpy as np
import easyocr
import torch
import argparse
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image, ImageEnhance

# --- 1. PREPROCESSING ---
def preprocess_image(image_path: str, max_angle=15.0) -> np.ndarray:
    img = cv2.imread("/content/test5.jpeg")
    if img is None:
        raise FileNotFoundError(f"Image not found at: {"/content/test5.jpeg"}")

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img.copy()
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = np.column_stack(np.where(binary > 0))

    if len(coords) < 10: return gray

    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45: angle = 90 + angle
    elif angle > 45: angle -= 90

    if abs(angle) > max_angle: return gray

    h, w = gray.shape
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=255)

# --- 2. LINE CLUSTERING ---
def group_into_lines(results, image_shape, y_tolerance=20, pad=2):
    """Groups scattered word bounding boxes into full horizontal lines using global margins."""
    if not results:
        return []

    boxes = []
    for bbox, _, _ in results:
        tl, tr, br, bl = bbox
        x_min, x_max = min(tl[0], bl[0]), max(tr[0], br[0])
        y_min, y_max = min(tl[1], tr[1]), max(bl[1], br[1])
        y_center = (y_min + y_max) / 2
        boxes.append({
            'x_min': x_min, 'x_max': x_max,
            'y_min': y_min, 'y_max': y_max,
            'y_center': y_center
        })

    boxes.sort(key=lambda b: b['y_center'])

    lines = []
    current_line = []
    current_y = None

    for b in boxes:
        if current_y is None:
            current_y = b['y_center']
            current_line.append(b)
        elif abs(b['y_center'] - current_y) <= y_tolerance:
            current_line.append(b)
            current_y = sum(x['y_center'] for x in current_line) / len(current_line)
        else:
            lines.append(current_line)
            current_line = [b]
            current_y = b['y_center']
    if current_line:
        lines.append(current_line)

    line_crops = []
    h, w = image_shape[:2]

    global_x_min = min(b['x_min'] for b in boxes)
    global_x_max = max(b['x_max'] for b in boxes)

    for line in lines:
        line_y_min = int(min(b['y_min'] for b in line))
        line_y_max = int(max(b['y_max'] for b in line))

        # Very strict padding to prevent grabbing adjacent lines
        y1 = max(0, line_y_min - pad)
        y2 = min(h, line_y_max + pad)

        x1 = max(0, int(global_x_min) - 5)
        x2 = min(w, int(global_x_max) + 5)

        line_crops.append((x1, y1, x2, y2))

    return line_crops

# --- 3. HYBRID OCR ENGINE ---
def hybrid_ocr(image: np.ndarray) -> str:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("Detecting text regions with EasyOCR...")
    reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
    img_rgb = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

    results = reader.readtext(img_rgb, paragraph=False)

    print("Clustering words into readable lines...")
    line_coords = group_into_lines(results, img_rgb.shape)

    print(f"Loading TrOCR model on {device}...")
    processor = TrOCRProcessor.from_pretrained('microsoft/trocr-large-handwritten')
    model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-large-handwritten').to(device)

    print("Reading cursive lines with TrOCR...")
    full_text = []

    for (x1, y1, x2, y2) in line_coords:
        line_crop = img_rgb[y1:y2, x1:x2]
        if line_crop.size == 0: continue

        pil_image = Image.fromarray(line_crop)

        # --- NEW: CONTRAST ENHANCEMENT ---
        # Deepen the purple ink so TrOCR can see it better
        enhancer = ImageEnhance.Contrast(pil_image)
        pil_image = enhancer.enhance(2.5)

        pixel_values = processor(pil_image, return_tensors="pt").pixel_values.to(device)
        with torch.no_grad():
            generated_ids = model.generate(pixel_values)

        text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

        # Filter out random hashtags or purely empty artifact reads
        if text.strip() and text.strip() != "#":
            full_text.append(text)
            print(f"Line read: {text}")

    return " ".join(full_text)

# --- MAIN EXECUTION ---
def main(image_path: str):
    try:
        processed_img = preprocess_image(image_path)
        final_text = hybrid_ocr(processed_img)

        print("\n" + "="*40)
        print("FINAL OCR OUTPUT")
        print("="*40)
        print(final_text)
        print("="*40 + "\n")

    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--image", type=str, default="/content/test5.jpeg")
    args, unknown = parser.parse_known_args()

    main(args.image)

Detecting text regions with EasyOCR...
Clustering words into readable lines...
Loading TrOCR model on cuda...


Loading weights:   0%|          | 0/635 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie decoder.model.decoder.embed_tokens.weight to decoder.output_projection.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-large-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Reading cursive lines with TrOCR...
Line read: Answer : They will take review of the make
Line read: customer boxed on that they will weaken
Line read: common #
Line read: can going the first and the time
Line read: these food quarks the quality of
Line read: business man will be used with make a
Line read: gerneywood may be " provide . and will make us .
Line read: # many #

FINAL OCR OUTPUT
Answer : They will take review of the make customer boxed on that they will weaken common # can going the first and the time these food quarks the quality of business man will be used with make a gerneywood may be " provide . and will make us . # many #

